In [1]:
# CELL 1: Imports & Config
import os
import json
import base64
import sqlite3
import asyncio
from datetime import datetime
from pathlib import Path

import nest_asyncio
from playwright.async_api import async_playwright
from openai import OpenAI
from dotenv import load_dotenv

# Fix for running asyncio in Jupyter Notebook
nest_asyncio.apply()

# Load .env file
load_dotenv()

# Config
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

SCREENSHOT_DIR = Path("screenshots")
OUTPUTS_DIR = Path("outputs")
SCREENSHOT_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Config loaded")
print(f"API Key found: {bool(OPENAI_API_KEY)}")
print(f"Screenshot folder: {SCREENSHOT_DIR}")
print(f"Outputs folder: {OUTPUTS_DIR}")


Config loaded
API Key found: True
Screenshot folder: screenshots
Outputs folder: outputs


In [6]:
# Cell 2: Database Setup

def init_db():
    conn = sqlite3.connect("qa_pipeline.db")
    cursor = conn.cursor()

    # Store each pipeline run 
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS runs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            url TEXT NOT NULL,
            username TEXT,
            status TEXT DEFAULT 'pending',
            created_at TEXT,
        )
        """
    )

    # Store each page crawled
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS pages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id INTEGER,
            url TEXT,
            screenshot_path TEXT,
            FOREIGN KEY (run_id) REFERENCES runs (id)
        )
        """
    )

    # Store each finding from the Vision Agent
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS findings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id INTEGER,
            page_url TEXT,
            issue_type TEXT,
            severity TEXT,
            FOREIGN KEY (run_id) REFERENCES runs (id)
        )
        """
    )

    # Stores the final generated report 
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS reports (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id INTEGER,
            content TEXT,
            created_at TEXT,
            FOREIGN KEY (run_id) REFERENCES runs (id)
        )
        """
    )

    conn.commit()
    conn.close()
    print("Database initialized")

init_db()

OperationalError: near ")": syntax error